# NL2Shell — Fine-tune Qwen3.5-0.8B on NL2Bash

In [ ]:
import subprocess
import sys

print("[0/6] Installing dependencies...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps",
    "trl", "peft", "accelerate", "bitsandbytes",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "datasets", "huggingface_hub",
])

In [ ]:
import os
from datasets import load_dataset, Dataset

# ── Constants ──────────────────────────────────────────────────────────────────
MODEL_NAME = "Qwen/Qwen3.5-0.8B"
OUTPUT_REPO = "AryaYT/nl2shell-0.8b"
MAX_SEQ_LENGTH = 512
HF_TOKEN = os.environ.get("HF_TOKEN", "")

SYSTEM_PROMPT = "You are an expert shell programmer. Given a natural language request, output ONLY the corresponding shell command. No explanations."

os.environ["WANDB_DISABLED"] = "true"

# ── macOS Synthetic Pairs ──────────────────────────────────────────────────────
MACOS_PAIRS = [
    ("list all installed homebrew packages", "brew list"),
    ("update homebrew and upgrade all packages", "brew update && brew upgrade"),
    ("show disk usage of current directory", "du -sh ."),
    ("find all Python files modified in the last 24 hours", "find . -name '*.py' -mtime -1"),
    ("show all running Docker containers", "docker ps"),
    ("kill the process using port 3000", "lsof -ti:3000 | xargs kill -9"),
    ("create a new git branch called feature-auth", "git checkout -b feature-auth"),
    ("show git log as one-line summaries", "git log --oneline -20"),
    ("compress the src directory into a tar.gz", "tar -czf src.tar.gz src/"),
    ("show system memory usage", "vm_stat | head -10"),
    ("list all open network connections", "lsof -i -P -n | head -20"),
    ("recursively find files larger than 100MB", "find / -size +100M -type f 2>/dev/null"),
    ("show the last 50 lines of the system log", "tail -50 /var/log/system.log"),
    ("restart the DNS cache on macOS", "sudo dscacheutil -flushcache && sudo killall -HUP mDNSResponder"),
    ("show all environment variables containing PATH", "env | grep PATH"),
    ("count lines of code in all TypeScript files", "find . -name '*.ts' | xargs wc -l"),
    ("check if port 8080 is in use", "lsof -i :8080"),
    ("show the top 10 largest files in current directory", "ls -lhS | head -10"),
    ("create an SSH tunnel from local 8080 to remote 80", "ssh -L 8080:localhost:80 user@host"),
    ("watch a directory for file changes", "fswatch -r . | head -20"),
    ("install a package globally with npm", "npm install -g package-name"),
    ("run a Python HTTP server on port 8000", "python3 -m http.server 8000"),
    ("show all cron jobs for current user", "crontab -l"),
    ("check SSL certificate expiry of a domain", "echo | openssl s_client -connect example.com:443 2>/dev/null | openssl x509 -noout -dates"),
    ("search for a string recursively in all files", "grep -r 'search_string' ."),
    ("show CPU and memory usage of all processes", "top -l 1 | head -20"),
    ("generate a random 32-character password", "openssl rand -base64 32"),
    ("download a file with curl and save it", "curl -LO https://example.com/file.tar.gz"),
    ("show the size of each subdirectory", "du -sh */ | sort -rh"),
    ("list all git branches sorted by last commit date", "git branch --sort=-committerdate"),
    ("set a file as executable", "chmod +x script.sh"),
    ("show the difference between two files", "diff file1.txt file2.txt"),
    ("find and delete all node_modules directories", "find . -name 'node_modules' -type d -prune -exec rm -rf {} +"),
    ("show which process is using the most CPU", "ps aux --sort=-%cpu | head -5"),
    ("create a symbolic link", "ln -s /path/to/original /path/to/link"),
    ("watch the output of a command every 2 seconds", "watch -n 2 'command'"),
    ("show all listening TCP ports", "netstat -tlnp 2>/dev/null || ss -tlnp"),
    ("convert a video to mp4 using ffmpeg", "ffmpeg -i input.mov -c:v libx264 -c:a aac output.mp4"),
    ("show git stash list", "git stash list"),
    ("run a command in the background and disown it", "nohup command &>/dev/null & disown"),
]

# ── Eval Test Prompts ──────────────────────────────────────────────────────────
EVAL_PROMPTS = [
    "list all files in the current directory",
    "find all Python files larger than 1MB",
    "show the last 20 lines of a log file",
    "create a compressed backup of the home directory",
    "check which process is using port 8080",
    "show all running processes sorted by memory usage",
    "count the number of lines in all .py files recursively",
]


def format_chatml(nl: str, cmd: str) -> str:
    """Format a single NL/CMD pair as ChatML."""
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{nl}<|im_end|>\n"
        f"<|im_start|>assistant\n{cmd}<|im_end|>"
    )


def get_dataset() -> Dataset:
    """Load NL2Bash + macOS synthetic pairs, formatted as ChatML text."""
    print("Loading NL2Bash dataset...")
    try:
        nl2bash = load_dataset("jiacheng-ye/nl2bash", split="train")
        nl_col, cmd_col = "nl", "bash"
        print(f"  jiacheng-ye/nl2bash: {len(nl2bash)} examples")
    except Exception as e:
        print(f"  jiacheng-ye/nl2bash failed: {e}, trying fallback...")
        nl2bash = load_dataset("AnonymousSub/NL2Bash", split="train")
        nl_col, cmd_col = "nl", "bash"
        print(f"  AnonymousSub/NL2Bash: {len(nl2bash)} examples")

    nl2bash_texts = []
    for row in nl2bash:
        nl = row[nl_col].strip()
        cmd = row[cmd_col].strip()
        if nl and cmd:
            nl2bash_texts.append(format_chatml(nl, cmd))

    macos_texts = [format_chatml(nl, cmd) for nl, cmd in MACOS_PAIRS]

    all_texts = nl2bash_texts + macos_texts
    dataset = Dataset.from_dict({"text": all_texts})
    dataset = dataset.shuffle(seed=42)
    print(f"Total training examples: {len(dataset)}")
    return dataset


def run_eval(model, tokenizer) -> None:
    """Run 7 test prompts through the model and print NL -> CMD."""
    print("\n" + "=" * 60)
    print("EVALUATION")
    print("=" * 60)

    for prompt in EVAL_PROMPTS:
        chatml_input = (
            f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
            f"<|im_start|>user\n{prompt}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
        inputs = tokenizer(chatml_input, return_tensors="pt").to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

        full_output = tokenizer.decode(outputs[0], skip_special_tokens=False)
        if "<|im_start|>assistant\n" in full_output:
            cmd = full_output.split("<|im_start|>assistant\n")[-1]
            cmd = cmd.split("<|im_end|>")[0].strip()
        else:
            cmd = tokenizer.decode(
                outputs[0][inputs["input_ids"].shape[1]:],
                skip_special_tokens=True,
            ).strip()

        print(f"  NL:  {prompt}")
        print(f"  CMD: {cmd}")
        print()

In [ ]:
from huggingface_hub import login
login(token=HF_TOKEN)

In [ ]:
import torch

print(f"[2/6] Loading {MODEL_NAME}...")

USE_UNSLOTH = True
try:
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )
    print("  Loaded with Unsloth FastLanguageModel (4-bit QLoRA)")

except Exception as e:
    print(f"  Unsloth failed: {e}")
    print("  Falling back to standard transformers + PEFT...")
    USE_UNSLOTH = False

    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    print("  Loaded with transformers + PEFT (4-bit QLoRA)")

model.print_trainable_parameters()

In [ ]:
print("[3/6] Preparing dataset...")
dataset = get_dataset()

In [ ]:
import math
from trl import SFTTrainer
from transformers import TrainingArguments

print("[4/6] Training...")

num_epochs = 3
batch_size = 8
grad_accum = 4
effective_batch = batch_size * grad_accum
steps_per_epoch = math.ceil(len(dataset) / effective_batch)
total_steps = steps_per_epoch * num_epochs
save_steps = max(steps_per_epoch // 2, 50)

print(f"  Examples:     {len(dataset)}")
print(f"  Epochs:       {num_epochs}")
print(f"  Batch:        {batch_size} x {grad_accum} = {effective_batch} effective")
print(f"  Steps/epoch:  {steps_per_epoch}")
print(f"  Total steps:  {total_steps}")
print(f"  Save every:   {save_steps} steps")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        warmup_steps=20,
        num_train_epochs=num_epochs,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        save_steps=save_steps,
        save_total_limit=2,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="/content/nl2shell-checkpoints",
        report_to="none",
    ),
)

# Mask user/system tokens — only train on assistant responses
try:
    from unsloth.chat_templates import train_on_responses_only
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )
    print("  train_on_responses_only applied")
except Exception as e:
    print(f"  train_on_responses_only skipped: {e}")

trainer_stats = trainer.train()
print(f"\nTraining complete!")
print(f"  Loss: {trainer_stats.training_loss:.4f}")
print(f"  Time: {trainer_stats.metrics['train_runtime']:.0f}s")

In [ ]:
print("[5/6] Running evaluation...")
if USE_UNSLOTH:
    FastLanguageModel.for_inference(model)
run_eval(model, tokenizer)

In [ ]:
from huggingface_hub import HfApi, create_repo

print("[6/6] Exporting and pushing to HuggingFace...")

api = HfApi()
create_repo(OUTPUT_REPO, private=False, exist_ok=True)

# Save merged 16-bit model
if USE_UNSLOTH:
    model.save_pretrained_merged(
        "/content/nl2shell-merged", tokenizer, save_method="merged_16bit"
    )

    # Export GGUF
    print("  Exporting GGUF q4_k_m...")
    try:
        model.save_pretrained_gguf(
            "/content/nl2shell-gguf-q4", tokenizer, quantization_method="q4_k_m"
        )
        print("    q4_k_m done")
    except Exception as e:
        print(f"    q4_k_m failed: {e}")

    print("  Exporting GGUF q8_0...")
    try:
        model.save_pretrained_gguf(
            "/content/nl2shell-gguf-q8", tokenizer, quantization_method="q8_0"
        )
        print("    q8_0 done")
    except Exception as e:
        print(f"    q8_0 failed: {e}")

    # Push merged model
    model.push_to_hub(OUTPUT_REPO, tokenizer=tokenizer)
    print("  Merged model pushed")

    # Push GGUF files
    for gguf_dir in ["/content/nl2shell-gguf-q4", "/content/nl2shell-gguf-q8"]:
        if os.path.exists(gguf_dir):
            for f in os.listdir(gguf_dir):
                if f.endswith(".gguf"):
                    api.upload_file(
                        path_or_fileobj=os.path.join(gguf_dir, f),
                        path_in_repo=f"gguf/{f}",
                        repo_id=OUTPUT_REPO,
                    )
                    print(f"    Uploaded {f}")
else:
    # Standard PEFT: push adapter, user merges later
    model.push_to_hub(OUTPUT_REPO)
    tokenizer.push_to_hub(OUTPUT_REPO)
    print("  LoRA adapter pushed (merge manually for GGUF)")

# Upload model card
MODEL_CARD = """---
license: mit
base_model: Qwen/Qwen3.5-0.8B
tags:
  - nl2bash
  - shell
  - terminal
  - command-line
  - qwen3.5
  - qlora
  - lecoder
  - cloudagi
datasets:
  - jiacheng-ye/nl2bash
language:
  - en
pipeline_tag: text-generation
---

# NL2Shell 0.8B — Natural Language to Shell Commands

Ultra-lightweight model for converting natural language to Unix/macOS shell commands.
Fine-tuned from Qwen3.5-0.8B using QLoRA on NL2Bash + 40 custom macOS command pairs.

## Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(\"AryaYT/nl2shell-0.8b\")
tokenizer = AutoTokenizer.from_pretrained(\"AryaYT/nl2shell-0.8b\")

prompt = \"<|im_start|>system\\nYou are an expert shell programmer. Given a natural language request, output ONLY the corresponding shell command. No explanations.<|im_end|>\\n<|im_start|>user\\nlist all files in the current directory<|im_end|>\\n<|im_start|>assistant\\n\"
inputs = tokenizer(prompt, return_tensors=\"pt\")
outputs = model.generate(**inputs, max_new_tokens=64)
print(tokenizer.decode(outputs[0], skip_special_tokens=False))
```

### With Ollama (GGUF)
```bash
# Q4_K_M (~400MB) for Raspberry Pi / edge
# Q8_0 (~650MB) for Mac / desktop
ollama run hf.co/AryaYT/nl2shell-0.8b
```

## Training Details
- **Base:** Qwen/Qwen3.5-0.8B
- **Method:** QLoRA (rank 16, alpha 32, all linear layers)
- **Data:** NL2Bash + 40 macOS synthetic pairs
- **Epochs:** 3
- **Hardware:** Google Colab A100
- **Built by:** [Arya Teja](https://aryateja.com) | [CloudAGI](https://cloudagi.ai)
"""

api.upload_file(
    path_or_fileobj=MODEL_CARD.encode(),
    path_in_repo="README.md",
    repo_id=OUTPUT_REPO,
)
print("  Model card uploaded")

print(f"\nModel live at: https://huggingface.co/{OUTPUT_REPO}")

In [ ]:
print("TRAINING_COMPLETE")